<a href="https://colab.research.google.com/github/trankhanhnamhoccode/Exact2026/blob/demo_learning/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cập nhật danh sách gói (bắt buộc)
!sudo apt-get update

# Cài đặt công cụ zstd để giải nén file .tar.zst của Ollama phiên bản mới
!sudo apt-get install -y zstd

# Cài đặt các thư viện Python
%pip install fastapi uvicorn ollama

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [91.2 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Get:13 http://securit

In [ ]:
# Chạy script cài đặt Ollama (lúc này đã có zstd để giải nén)
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%################                                                       27.3%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

# (Tùy chọn) Kill tiến trình ollama cũ nếu có, để tránh xung đột cổng
subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

# Khởi động ollama serve trong background
# stdout và stderr được chuyển hướng để tránh làm lộn xộn output của notebook
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi vài giây cho server khởi động hoàn toàn
time.sleep(4)
print("✅ Ollama server đã được khởi động thành công!")


✅ Ollama server đã được khởi động thành công!


In [ ]:
import ollama
ollama.pull('hf.co/bartowski/Llama-3.1_OpenScholar-8B-GGUF:Q6_K')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [ ]:
# Kiểm tra xem server đã sẵn sàng chưa




# Tải model Qwen3:8b về (chỉ cần làm một lần)
!ollama pull qwen3:0.6b
!ollama pull qwen3:1.7b
!ollama pull qwen3:4b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 7f4030143c1c:   4% ▕                  ▏  22 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  19% ▕███               ▏  97 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  27% ▕████              ▏ 139 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  39% ▕███████           ▏ 203 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  53% ▕█████████         ▏ 275 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  60% ▕██████████        ▏ 313 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  74% ▕█████████████     ▏ 387 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  81% ▕██████████████    ▏ 422 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  84% ▕███████████████   ▏ 441 MB/522 MB    

In [ ]:
!ollama list

]11;?\NAME          ID              SIZE      MODIFIED      
qwen3:4b      359d7dd4bcda    2.5 GB    2 minutes ago    
qwen3:1.7b    8f68893c685c    1.4 GB    3 minutes ago    
qwen3:0.6b    7df6b6e09427    522 MB    3 minutes ago    


In [ ]:
# Cell 5: Hàm gọi model (thay thế cho API)
from ollama import generate
from pydantic import BaseModel
from pathlib import Path

class Response(BaseModel):
    qtype: str


def load_prompt(filename: str,path: str, **kwargs) -> str:
    prompt_path = Path(path+ filename)
    template = prompt_path.read_text(encoding="utf-8")
    # Nếu có placeholder, thay thế bằng kwargs
    for key, value in kwargs.items():
        placeholder = "{{" + key + "}}"
        template = template.replace(placeholder, value)
    return template


def classify_with_llm(question: str) :
    prompt_text = load_prompt(filename="prompt.txt",path="/kaggle/input/datasets/nam1706/prompt-v1-0/", QUESTION="QUESTION: " + question)
    response = generate(
        model='qwen3:0.6b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'think':False,
            'stream':False,
            'format': Response.model_json_schema(),
            'keep_alive':'5m',
            'num_predict': 1000  # CHẶN: Không cho sinh quá 100 token
        }
    )
    # ... gọi ollama với prompt trên, temperature=0
    raw = response # giả sử đây là string trả về
    # LLM thường trả về markdown json```json\n{...}\n```, cần làm sạch
    # Dùng regex tìm khối JSON đầu tiên
    return raw['response']

In [ ]:

q0= "Based on the premises, what can we conclude about the curriculum?\nA. It enhances student engagement but not critical thinking\nB. It enhances critical thinking\nC. It needs more resources to enhance critical thinking\nD. It is well-structured but lacks exercises"+"\nDoes the combination of faculty priorities and curriculum features lead to enhanced critical thinking?"

q1 = "Two electric charges q1 = +9×10⁻⁶ C and q2 = -9×10⁻⁶ C are placed 10 cm apart. A charge q3 = +9×10⁻⁶ C is at the midpoint. Bring the problem to SMT formula to find forces between q1 and q3"

q2 = "Calculate the energy stored in capacitor C when C = 100 μF and U = 30 V."

q3 = "calculate "

q4 = "Calculate the current through a 10 Ohm resistor connected in series with a 20 Ohm resistor to a 12V battery."
print(classify_with_llm(q1))


{
  "qtype": "physics"
}


In [ ]:
from ollama import generate
from pydantic import BaseModel
from pathlib import Path



def problem_extracter(question: str):
    prompt_text = load_prompt(filename="phyExtractingProblemPrompt.txt",path="/kaggle/input/datasets/nam1706/phyprompt-v1-3/", QUESTION="QUESTION: " + question)
    response = generate(
        model='qwen3:1.7b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'repeat_penalty':1.2,
            'keep_alive':'5m',
            'num_predict': 6000  # CHẶN: Không cho sinh quá 100 token
        }
    )
    raw = response
    return raw['response']



In [ ]:
q2 = "Calculate the energy stored in capacitor C when C = 100 μF and U = 30 V."
q0 = "An object undergoes simple harmonic motion with the equation x = 4cos(2πt + π/2) cm. Determine the amplitude, period, and initial position of the object."
q3_diff2 = "In a circuit, a 10 Ohm resistor (R1) and a 15 Ohm resistor (R2) are connected in series. This combination is then connected in series with a 4 Ohm resistor (R3) and a 24 V battery. Calculate the total current flowing from the battery and the voltage drop across R2."
q6 = "A 5 Ohm and a 10 Ohm resistor are connected in series to a 15 V battery. Find the current."
q4 = "In a circuit, a 6 Ohm resistor (R1) and a 12 Ohm resistor (R2) are connected in parallel. This combination is then connected in series with a 10 Ohm resistor (R3) and a 36 V battery. Calculate the total current and the power dissipated by R3."
q5 = "A car accelerates uniformly from rest to 20 m/s in 8 seconds. It then accelerates again at 1.5 m/s² for 6 seconds. Calculate the final velocity of the car and the total distance traveled."
q7 = "In a circuit, a 6 Ohm (R1) and a 12 Ohm (R2) are in parallel. This block is in series with a 10 Ohm (R3) and a 36 V source. Find the current through the source and the voltage across R2."
q8 = "How much heat is needed to heat 300 g of aluminum from 25°C to 75°C? Specific heat of aluminum is 900 J/(kg·K)."
q9 = "A stone is dropped from rest. How far does it fall in 4 seconds? (Take g = 10 m/s)"
q10 = "A 5 kg block on a frictionless 30° incline is connected to a 3 kg hanging mass by a string over a pulley. Find the acceleration and the tension in the string."

q11 = "In a circuit, a 10 V battery (E1) with internal resistance 1 Ohm, and a 6 V battery (E2) with internal resistance 0.5 Ohm are connected in parallel. The combination powers a load resistor R = 10 Ohm. Find the current through the load resistor."
q12 = "In a circuit, a 12 V battery (E1) with an internal resistance of 2 Ω, and a 6 V battery (E2) with an internal resistance of 1 Ω are connected in parallel. This parallel combination of batteries is then used to power a load resistor R = 10 Ω connected across the combination. Determine the current flowing through the load resistor R, and the current supplied by each battery."

q13 ="A 5 kg block lies on a rough 37° incline (coefficient of kinetic friction µ_k = 0.2). This block is connected by a light string passing over a frictionless pulley to a hanging 8 kg mass. The system is released from rest. Determine the acceleration of the system and the tension in the string. (Take g = 10 m/s², sin37° = 0.6, cos37° = 0.8)."
q14 = "A 0.5 kg iron block at 300°C is dropped into 2 kg of water at 25°C. Assuming no heat loss, find the final equilibrium temperature. (c_iron = 450 J/kg·K, c_water = 4186 J/kg·K)"
q15 = "A car accelerates uniformly from rest to 20 m/s in 8 seconds. It then accelerates again at 1.5 m/s² for 6 seconds. Calculate the final velocity of the car and the total distance traveled."
q16 = "An object undergoes simple harmonic motion with maximum speed vmax = 16π (cm/s) and maximum acceleration amax = 6.4 (m/s²). Take π² = 10. Calculate the period and frequency of oscillation."


In [ ]:
import json
import re
import time
import ollama
import numpy as np
from typing import List, Dict, Optional, Tuple, Any
from sentence_transformers import SentenceTransformer
from sympy import symbols, Eq, solve, sympify, Symbol
from sympy.parsing.sympy_parser import parse_expr

# ============================================================
# 1. Load Embedding Model
# ============================================================
try:
    EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
except Exception as e:
    print(f"Warning: Could not load embedding model: {e}")
    EMBED_MODEL = None

def retrieve_relations(question: str, domain: str, top_k: int = 5) -> List[Dict]:
    """Trả về danh sách các relation liên quan dựa trên embedding similarity."""
    candidates = FORMULA_STORE.get(domain, [])
    if not candidates or EMBED_MODEL is None:
        return candidates[:top_k]  # fallback

    candidate_texts = [
        f"{r['name']}. {r['use']}. Keywords: {' '.join(r.get('keywords', []))}"
        for r in candidates
    ]
    question_emb = EMBED_MODEL.encode([question])[0]
    candidate_embs = EMBED_MODEL.encode(candidate_texts)
    similarities = np.dot(candidate_embs, question_emb)
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [candidates[i] for i in top_indices]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
q7 = "In a circuit, a 6 Ohm (R1) and a 12 Ohm (R2) are in series. This block is in paralel with a 10 Ohm (R3) and a 36 V source. Find the current through the source and the voltage across R2."




In [ ]:
BASIC_ARITHMETIC_TOOLS = [
    {
        "id": "add",
        "name": "Addition",
        "use": "Calculate sum of two quantities",
        "type": "arithmetic",
        "equation": "Eq(sum, x + y)",
        "variables": ["sum", "x", "y"],
        "keywords": ["add", "sum", "total", "plus"]
    },
    {
        "id": "subtract",
        "name": "Subtraction",
        "use": "Calculate difference between two quantities",
        "type": "arithmetic",
        "equation": "Eq(diff, x - y)",
        "variables": ["diff", "x", "y"],
        "keywords": ["subtract", "difference", "minus"]
    },
    {
        "id": "multiply",
        "name": "Multiplication",
        "use": "Calculate product of two quantities",
        "type": "arithmetic",
        "equation": "Eq(prod, x * y)",
        "variables": ["prod", "x", "y"],
        "keywords": ["multiply", "product", "times"]
    },
    {
        "id": "divide",
        "name": "Division",
        "use": "Calculate quotient of two quantities",
        "type": "arithmetic",
        "equation": "Eq(quot, x / y)",
        "variables": ["quot", "x", "y"],
        "keywords": ["divide", "quotient", "ratio"]
    }
]

In [ ]:
FORMULA_STORE = {
    "electricity": [
        {
            "id": "ohms_law",
            "name": "Ohm's Law",
            "use": "Calculate one of V, I, R from the other two",
            "type": "formula",
            "equation": "Eq(V, I * R)",
            "variables": ["V", "I", "R"],
            "keywords": ["ohm", "current", "voltage", "resistance", "V", "I", "R"]
        },
        {
            "id": "power_law",
            "name": "Joule–Lenz / Power Law",
            "use": "Calculate power P from V and I (or equivalent forms)",
            "type": "formula",
            "equation": "Eq(P, V * I)",
            "variables": ["P", "V", "I"],
            "keywords": ["power", "watt", "P", "V", "I"]
        },
        {
            "id": "series_resistance",
            "name": "Resistors in Series",
            "use": "Find total resistance for resistors connected in series",
            "type": "formula",
            "equation": "Eq(R_total, R1 + R2)",
            "variables": ["R_total", "R1", "R2"],
            "keywords": ["series", "total resistance", "R_total", "connected in series"]
        },
        {
            "id": "parallel_resistance",
            "name": "Resistors in Parallel",
            "use": "Find total resistance for resistors connected in parallel",
            "type": "formula",
            "equation": "Eq(1/R_total, 1/R1 + 1/R2)",
            "variables": ["R_total", "R1", "R2"],
            "keywords": ["parallel", "total resistance", "R_total", "connected in parallel"]
        },
        {
            "id": "capacitor_energy",
            "name": "Energy stored in a capacitor",
            "use": "Calculate energy E from capacitance C and voltage U",
            "type": "formula",
            "equation": "Eq(E, 0.5 * C * U**2)",
            "variables": ["E", "C", "U"],
            "keywords": ["capacitor", "energy", "E", "C", "U", "voltage"]
        },
        {
            "id": "coulomb_force",
            "name": "Coulomb's Law",
            "use": "Calculate electrostatic force between two point charges",
            "type": "formula",
            "equation": "Eq(F, k * q1 * q2 / r**2)",
            "variables": ["F", "k", "q1", "q2", "r"],
            "keywords": ["coulomb", "force", "charge", "electrostatic", "F", "q", "r"]
        },
        {
            "id": "current_division",
            "name": "Current Divider Rule",
            "use": "Find current through a branch in a parallel circuit",
            "type": "formula",
            "equation": "Eq(I_x, I_total * (R_total / R_x))",
            "variables": ["I_x", "I_total", "R_total", "R_x"],
            "keywords": ["current division", "parallel", "current", "branch"]
        },
        {
            "id": "voltage_division",
            "name": "Voltage Divider Rule",
            "use": "Find voltage across a resistor in a series circuit",
            "type": "formula",
            "equation": "Eq(V_x, V_total * (R_x / R_total))",
            "variables": ["V_x", "V_total", "R_x", "R_total"],
            "keywords": ["voltage division", "series", "voltage", "resistor"]
        },
        {
            "id": "electric_power_alternate",
            "name": "Power (alternate forms)",
            "use": "Calculate power using P = I^2 R or P = V^2 / R",
            "type": "formula",
            "equation": "Eq(P, I**2 * R)",
            "variables": ["P", "I", "R"],
            "keywords": ["power", "I^2 R", "Joule heating", "P", "I", "R"]
        }
        # Note: power_law already covers P=V*I; we might add the other forms as separate relations.
        # Chúng ta có thể thêm các dạng khác của công suất vào cùng một quan hệ bằng cách sử dụng một hàm linh hoạt,
        # nhưng để embedding dễ dàng, ta nên tách chúng ra.
    ],
    "kinematics": [
        {
            "id": "velocity_acceleration_time",
            "name": "v = u + at",
            "use": "Calculate final velocity v from initial velocity u, acceleration a, and time t",
            "type": "formula",
            "equation": "Eq(v, u + a*t)",
            "variables": ["v", "u", "a", "t"],
            "keywords": ["velocity", "acceleration", "time", "motion", "v", "u", "a", "t"]
        },
        {
            "id": "displacement_uvt",
            "name": "s = (u + v)t / 2",
            "use": "Calculate displacement s from initial and final velocities and time",
            "type": "formula",
            "equation": "Eq(s, (u + v) * t / 2)",
            "variables": ["s", "u", "v", "t"],
            "keywords": ["displacement", "initial velocity", "s", "u", "v", "t","final velocity","distance"]
        },
        {
            "id": "displacement_ut_half_at2",
            "name": "s = ut + ½at²",
            "use": "Calculate displacement s from initial velocity, acceleration, and time",
            "type": "formula",
            "equation": "Eq(s, u*t + 0.5*a*t**2)",
            "variables": ["s", "u", "a", "t"],
            "keywords": ["displacement", "acceleration", "time", "s", "u", "a", "t","distance"]
        },
        {
            "id": "velocity_squared",
            "name": "v² = u² + 2as",
            "use": "Relate velocities, acceleration, and displacement",
            "type": "formula",
            "equation": "Eq(v**2, u**2 + 2*a*s)",
            "variables": ["v", "u", "a", "s"],
            "keywords": ["velocity squared", "displacement", "acceleration", "v^2", "u^2"]
        },
        {
            "id": "average_speed",
            "name": "Average speed",
            "use": "Calculate average speed from total distance and total time",
            "type": "formula",
            "equation": "Eq(v_avg, s_total / t_total)",
            "variables": ["v_avg", "s_total", "t_total"],
            "keywords": ["average speed", "total distance", "total time", "v_avg"]
        },
        {
            "id": "shm_period_angular_freq",
            "name": "T = 2π/ω",
            "use": "Relate period T and angular frequency ω in SHM",
            "type": "formula",
            "equation": "Eq(T, 2*pi/omega)",
            "variables": ["T", "omega", "pi"],
            "keywords": ["period", "angular frequency", "SHM", "T", "omega", "pi"]
        },
        {
            "id": "shm_frequency_period",
            "name": "f = 1/T",
            "use": "Relate frequency f and period T",
            "type": "formula",
            "equation": "Eq(f, 1/T)",
            "variables": ["f", "T"],
            "keywords": ["frequency", "period", "SHM", "f", "T"]
        },
        {
            "id": "shm_max_speed",
            "name": "v_max = ωA",
            "use": "Calculate maximum speed in SHM from amplitude A and angular frequency ω",
            "type": "formula",
            "equation": "Eq(v_max, omega * A)",
            "variables": ["v_max", "omega", "A"],
            "keywords": ["maximum speed", "SHM", "amplitude", "angular frequency", "v_max", "omega", "A"]
        },
        {
            "id": "shm_max_acceleration",
            "name": "a_max = ω²A",
            "use": "Calculate maximum acceleration in SHM",
            "type": "formula",
            "equation": "Eq(a_max, omega**2 * A)",
            "variables": ["a_max", "omega", "A"],
            "keywords": ["maximum acceleration", "SHM", "a_max", "omega", "A"]
        },

    ],
    "dynamics": [
        {
            "id": "newton2",
            "name": "Newton's Second Law",
            "use": "Calculate net force F from mass m and acceleration a",
            "type": "formula",
            "equation": "Eq(F_net, m * a)",
            "variables": ["F_net", "m", "a"],
            "keywords": ["newton", "force", "mass", "acceleration", "F", "m", "a"]
        },
        {
            "id": "weight",
            "name": "Weight",
            "use": "Calculate weight W from mass m and gravity g",
            "type": "formula",
            "equation": "Eq(W, m * g)",
            "variables": ["W", "m", "g"],
            "keywords": ["weight", "gravity", "W", "m", "g"]
        },
        {
            "id": "friction_kinetic",
            "name": "Kinetic Friction",
            "use": "Calculate kinetic friction force f_k from normal force N and coefficient μ_k",
            "type": "formula",
            "equation": "Eq(f_k, mu_k * N)",
            "variables": ["f_k", "mu_k", "N"],
            "keywords": ["friction", "kinetic", "f_k", "mu_k", "N"]
        },
        {
            "id": "friction_static_max",
            "name": "Maximum Static Friction",
            "use": "Calculate maximum static friction force f_s_max",
            "type": "formula",
            "equation": "Eq(f_s_max, mu_s * N)",
            "variables": ["f_s_max", "mu_s", "N"],
            "keywords": ["static friction", "maximum", "f_s_max", "mu_s", "N"]
        },
        {
            "id": "gravity_force",
            "name": "Newton's Law of Gravitation",
            "use": "Calculate gravitational force between two masses",
            "type": "formula",
            "equation": "Eq(F, G * m1 * m2 / r**2)",
            "variables": ["F", "G", "m1", "m2", "r"],
            "keywords": ["gravity", "gravitational force", "F", "G", "m1", "m2", "r"]
        },
        {
            "id": "spring_force",
            "name": "Hooke's Law",
            "use": "Calculate spring force from spring constant k and displacement x",
            "type": "formula",
            "equation": "Eq(F, -k * x)",
            "variables": ["F", "k", "x"],
            "keywords": ["spring", "Hooke", "force", "k", "x"]
        },
        {
            "id": "momentum",
            "name": "Momentum",
            "use": "Calculate momentum p from mass m and velocity v",
            "type": "formula",
            "equation": "Eq(p, m * v)",
            "variables": ["p", "m", "v"],
            "keywords": ["momentum", "p", "m", "v"]
        },
        {
            "id": "impulse",
            "name": "Impulse-Momentum Theorem",
            "use": "Relate impulse to change in momentum",
            "type": "formula",
            "equation": "Eq(F_avg * delta_t, m * (v_f - v_i))",
            "variables": ["F_avg", "delta_t", "m", "v_f", "v_i"],
            "keywords": ["impulse", "momentum", "force", "time"]
        },
        {
            "id": "work_constant_force",
            "name": "Work done by constant force",
            "use": "Calculate work W from force F, displacement s, and angle θ",
            "type": "formula",
            "equation": "Eq(W, F * s * cos(theta))",
            "variables": ["W", "F", "s", "theta"],
            "keywords": ["work", "force", "displacement", "angle", "W"]
        },
        {
            "id": "kinetic_energy",
            "name": "Kinetic Energy",
            "use": "Calculate kinetic energy KE from mass m and velocity v",
            "type": "formula",
            "equation": "Eq(KE, 0.5 * m * v**2)",
            "variables": ["KE", "m", "v"],
            "keywords": ["kinetic energy", "KE", "m", "v"]
        },
        {
            "id": "potential_energy_gravity",
            "name": "Gravitational Potential Energy",
            "use": "Calculate PE from mass m, gravity g, and height h",
            "type": "formula",
            "equation": "Eq(PE, m * g * h)",
            "variables": ["PE", "m", "g", "h"],
            "keywords": ["potential energy", "gravity", "height", "PE", "m", "g", "h"]
        },
        {
            "id": "power_work_time",
            "name": "Power (work/time)",
            "use": "Calculate power P from work W and time t",
            "type": "formula",
            "equation": "Eq(P, W / t)",
            "variables": ["P", "W", "t"],
            "keywords": ["power", "work", "time", "P", "W", "t"]
        }
    ],
    "thermodynamics": [
        {
            "id": "heat_transfer",
            "name": "Heat Transfer (Q=mcΔT)",
            "use": "Calculate heat Q from mass m, specific heat c, and temperature change ΔT",
            "type": "formula",
            "equation": "Eq(Q, m * c * (T_f - T_i))",
            "variables": ["Q", "m", "c", "T_f", "T_i"],
            "keywords": ["heat", "specific heat", "temperature", "Q", "m", "c", "T"]
        },
        {
            "id": "latent_heat",
            "name": "Latent Heat",
            "use": "Calculate heat Q for phase change from mass m and latent heat L",
            "type": "formula",
            "equation": "Eq(Q, m * L)",
            "variables": ["Q", "m", "L"],
            "keywords": ["latent heat", "phase change", "Q", "m", "L"]
        },
        {
            "id": "thermal_equilibrium",
            "name": "Thermal Equilibrium (no phase change)",
            "use": "Find final temperature when two substances reach thermal equilibrium",
            "type": "theory",
            "equation": "Eq(m1 * c1 * (T_f - T1) + m2 * c2 * (T_f - T2), 0)",
            "variables": ["m1", "c1", "T1", "m2", "c2", "T2", "T_f"],
            "keywords": ["thermal equilibrium", "calorimetry", "final temperature", "mixing"]
        },
        {
            "id": "ideal_gas_law",
            "name": "Ideal Gas Law",
            "use": "Relate pressure P, volume V, amount n, and temperature T for an ideal gas",
            "type": "formula",
            "equation": "Eq(P * V, n * R * T)",
            "variables": ["P", "V", "n", "R", "T"],
            "keywords": ["ideal gas", "pressure", "volume", "moles", "temperature", "R"]
        }
    ],
    "shm": [
        # SHM formulas are mostly in kinematics, but we can add more specific ones here.
        {
            "id": "shm_position",
            "name": "Displacement in SHM",
            "use": "Calculate position x from amplitude A, angular frequency ω, time t, and phase φ",
            "type": "formula",
            "equation": "Eq(x, A * cos(omega * t + phi))",
            "variables": ["x", "A", "omega", "t", "phi"],
            "keywords": ["SHM", "position", "displacement", "x", "A", "omega", "t", "phi"]
        },
        {
            "id": "shm_velocity",
            "name": "Velocity in SHM",
            "use": "Calculate velocity v in SHM",
            "type": "formula",
            "equation": "Eq(v, -A * omega * sin(omega * t + phi))",
            "variables": ["v", "A", "omega", "t", "phi"],
            "keywords": ["SHM", "velocity", "v", "omega", "A"]
        },
        {
            "id": "shm_acceleration",
            "name": "Acceleration in SHM",
            "use": "Calculate acceleration a in SHM",
            "type": "formula",
            "equation": "Eq(a, -omega**2 * x)",
            "variables": ["a", "omega", "x"],
            "keywords": ["SHM", "acceleration", "a", "omega", "x"]
        },
        {
            "id": "shm_spring_period",
            "name": "Period of a mass-spring system",
            "use": "Calculate period T from mass m and spring constant k",
            "type": "formula",
            "equation": "Eq(T, 2*pi * sqrt(m/k))",
            "variables": ["T", "m", "k", "pi"],
            "keywords": ["spring", "period", "SHM", "T", "m", "k"]
        },
        {
            "id": "shm_pendulum_period",
            "name": "Period of a simple pendulum",
            "use": "Calculate period T from length L and gravity g",
            "type": "formula",
            "equation": "Eq(T, 2*pi * sqrt(L/g))",
            "variables": ["T", "L", "g", "pi"],
            "keywords": ["pendulum", "period", "SHM", "T", "L", "g"]
        }
    ]
    # Can add more domains like "optics", "waves", etc. as needed.
}

In [ ]:
import json
import re
import sympy as sp
from sympy import symbols, Eq, solve, sympify

# ========== TRÍCH XUẤT BIẾN TỪ CHUỖI PHƯƠNG TRÌNH ==========
def extract_variables(eq_str):
    """Lấy danh sách tên biến (symbol) từ chuỗi phương trình Eq(...)."""
    # Tách chuỗi trong Eq(...) rồi dùng sympy parsing nhẹ hoặc regex
    # Cách đơn giản: tìm tất cả các từ gồm chữ cái và số sau dấu gạch dưới (không phải từ khóa)
    # Nhưng để an toàn với sympy, ta dùng sympy.sympify để parse rồi lấy .free_symbols
    try:
        # Tạm thay thế các hàm như sin, cos, pi
        expr = sympify(eq_str)
        if isinstance(expr, Eq):
            all_syms = expr.lhs.free_symbols | expr.rhs.free_symbols
        else:
            all_syms = expr.free_symbols
        return [str(s) for s in all_syms]
    except:
        # fallback regex: lấy các từ có chữ cái
        return re.findall(r'[a-zA-Z_]\w*', eq_str)

# ========== FORWARD SOLVE (BFS TRÊN PHƯƠNG TRÌNH) ==========
def forward_solve(given, unknown, equations):
    """
    given: list of dicts with 'symbol', 'value'
    unknown: list of dicts with 'symbol'
    equations: list of dicts with 'equation' and 'description' (or 'id')
    """
    # Tạo symbol dictionary cho tất cả biến
    all_var_names = set()
    for g in given:
        all_var_names.add(g['symbol'])
    for u in unknown:
        all_var_names.add(u['symbol'])
    for eq_obj in equations:
        vs = extract_variables(eq_obj['equation'])
        for v in vs:
            all_var_names.add(v)

    symbol_dict = {}
    for name in all_var_names:
        if name == 'pi':
            symbol_dict[name] = sp.pi
        else:
            symbol_dict[name] = symbols(name)

    # Parse các phương trình
    parsed_eqs = []
    for eq_obj in equations:
        eq_str = eq_obj['equation']
        desc = eq_obj.get('description', '')
        try:
            eq = sympify(eq_str, locals=symbol_dict)
            if not isinstance(eq, Eq):
                eq = Eq(eq, 0)
            parsed_eqs.append({
                'eq': eq,
                'variables': [str(v) for v in (eq.lhs.free_symbols | eq.rhs.free_symbols)],
                'id': desc
            })
        except Exception as e:
            print(f"Lỗi parse {eq_str}: {e}")

    # Khởi tạo known
    known_values = {}
    for g in given:
        try:
            known_values[g['symbol']] = float(g['value'])
        except:
            continue

    steps = []
    changed = True
    while changed:
        changed = False
        for eq_info in parsed_eqs:
            eq = eq_info['eq']
            var_names = eq_info['variables']
            unknown_vars = [v for v in var_names if v not in known_values]
            known_vars = [v for v in var_names if v in known_values]
            # Nếu chỉ còn 1 ẩn -> có thể giải
            if len(unknown_vars) == 1:
                target = unknown_vars[0]
                subs_dict = {symbol_dict[v]: known_values[v] for v in known_vars}
                try:
                    sol = solve(eq.subs(subs_dict), symbol_dict[target])
                    if sol:
                        val = float(sol[0].evalf())
                        known_values[target] = val
                        steps.append({
                            'tool_id': eq_info['id'],
                            'target': target,
                            'value': val,
                            'inputs': {v: known_values[v] for v in known_vars}
                        })
                        changed = True
                        break  # restart quét từ đầu
                except:
                    continue

    # Kiểm tra unknown cuối cùng
    final_vals = {}
    missing = []
    for u in unknown:
        sym = u['symbol']
        if sym in known_values:
            final_vals[sym] = known_values[sym]
        else:
            missing.append(sym)

    if missing:
        print(f"Không thể tìm được các biến: {missing}")
        return None, None
    return steps, final_vals

# ========== CHẠY VÍ DỤ VỚI BÀI KINEMATICS ==========
if __name__ == "__main__":
    # JSON đầu ra từ LLM (đã có sẵn)
    q7 = "A capacitor with capacitance C = 10 µF is connected to an ideal LC circuit. When the voltage across the capacitor is 100√2 V, what is the electric field energy?"
    print(q7)
    json_data = problem_extracter(q7)

    data = json.loads(json_data)
    given = data['given']
    unknown = data['unknown']
    equations = data['equations']

    print("Giải bài toán...")
    steps, values = forward_solve(given, unknown, equations)

    if steps:
        print("\n=== Các bước suy luận ===")
        for i, s in enumerate(steps, 1):
            print(f"{i}. {s['tool_id']}: {s['target']} = {s['value']:.2f}  (inputs: {s['inputs']})")
        print("\n=== Kết quả cuối ===")
        for sym, val in values.items():
            print(f"{sym} = {val}")
    else:
        print("Không giải được.")

A capacitor with capacitance C = 10 µF is connected to an ideal LC circuit. When the voltage across the capacitor is 100√2 V, what is the electric field energy?


JSONDecodeError: Expecting ',' delimiter: line 4 column 52 (char 137)

In [ ]:
problem_extracter(q7)

'{\n  "given": [\n    {"name": "capacitance", "symbol": "C", "value": 10, "unit": "µF"},\n    {"name": "voltage", "symbol": "V", "value": 100*1.4142, "unit": "V"}\n  ],\n  "unknown": [\n    {"name": "electric field energy", "symbol": "U", "value": null, "unit": "J"}\n  ],\n  "topology": "electricity",\n  "equations": [\n    {\n      "equation": "Eq(U, 0.5 * C * V**2)",\n      "description": "Electric field energy in capacitor: U = ½ C V²"\n    }\n  ],\n  "problem_type": "electricity",\n  "difficulty": 1\n}'

In [ ]:
import json
import re
import sympy as sp
from sympy import symbols, Eq, solve, sympify

# ============ CÁC HÀM CÓ SẴN (giữ nguyên forward_solve) ============
def forward_solve(given, unknown, tools):
    all_var_names = set()
    for g in given: all_var_names.add(g['symbol'])
    for u in unknown: all_var_names.add(u['symbol'])
    for tool in tools:
        for v in tool['variables']:
            all_var_names.add(v)
    symbol_dict = {}
    for name in all_var_names:
        if name == 'pi':
            symbol_dict[name] = sp.pi
        else:
            symbol_dict[name] = symbols(name)

    equations = []
    for tool in tools:
        try:
            eq = sympify(tool['equation'], locals=symbol_dict)
            if not isinstance(eq, Eq):
                eq = Eq(eq, 0)
            equations.append({
                'eq': eq,
                'variables': [symbol_dict[v] for v in tool['variables']],
                'id': tool['id']
            })
        except Exception as e:
            print(f"Lỗi parse {tool['equation']}: {e}")

    known_values = {}
    for g in given:
        try:
            known_values[g['symbol']] = float(g['value'])
        except:
            continue

    steps = []
    changed = True
    while changed:
        changed = False
        for eq_info in equations:
            eq = eq_info['eq']
            var_names = [str(v) for v in eq_info['variables']]
            unknown_vars = [v for v in var_names if v not in known_values]
            known_vars = [v for v in var_names if v in known_values]
            if len(unknown_vars) == 1:
                target = unknown_vars[0]
                subs_dict = {symbol_dict[v]: known_values[v] for v in known_vars}
                try:
                    sol = solve(eq.subs(subs_dict), symbol_dict[target])
                    if sol:
                        val = float(sol[0].evalf())
                        known_values[target] = val
                        steps.append({
                            'tool_id': eq_info['id'],
                            'target': target,
                            'value': val,
                            'inputs': {v: known_values[v] for v in known_vars}
                        })
                        changed = True
                        break
                except:
                    continue

    final_vals = {}
    missing = []
    for u in unknown:
        if u['symbol'] in known_values:
            final_vals[u['symbol']] = known_values[u['symbol']]
        else:
            missing.append(u['symbol'])
    if missing:
        print(f"Không thể tìm được các biến: {missing}")
        return None, None
    return steps, final_vals

# ============ HÀM INSTANTIATE TỔNG QUÁT MỚI ============
def instantiate_general(given, unknown, connections, tools):
    """
    Sinh phương trình cụ thể từ connections (danh sách chuỗi) và bộ tools.
    """
    used_names = set()
    for g in given: used_names.add(g['symbol'])
    for u in unknown: used_names.add(u['symbol'])

    # Map từ string nhóm -> tên biến đại diện
    group_map = {}
    # Khởi tạo: mỗi biến đơn tự đại diện
    for g in given: group_map[g['symbol']] = g['symbol']
    for u in unknown: group_map[u['symbol']] = u['symbol']

    specific_eqs = []
    last_equivalent_var = None  # biến tổng trở / điện áp tương đương cuối cùng

    # 1. Xử lý connections kiểu motion
    if any('->' in c for c in connections):
        # Dùng instantiate_multistage cũ (bạn có thể giữ lại)
        # Nhưng để tổng quát, ta gọi hàm đó nếu cần. Ở đây ta giả định motion có connections riêng.
        return instantiate_multistage(tools, given, unknown, "multi_stage_linear_motion")

    # 2. Xử lý connections kiểu mạch điện (series/parallel)
    for conn_str in connections:
        if conn_str == "direct_formula":
            continue
        # Phân tích: <left> (series|parallel) <right>
        # Cho phép left/right là tên biến hoặc (nhóm)
        pattern = r'\(?([\w,]+)\)?\s*(series|parallel)\s*\(?([\w,]+)\)?'
        match = re.match(pattern, conn_str.strip())
        if not match:
            print(f"Không parse được connection: {conn_str}")
            continue

        left_str, op, right_str = match.groups()
        left_str = left_str.strip()
        right_str = right_str.strip()

        # Tìm biến đại diện cho left
        def resolve_var(part):
            part = part.strip()
            if ',' in part or '(' in part:  # nhóm
                key = part.replace('(', '').replace(')', '').replace(' ', '')
                return group_map.get(key)
            else:
                if part not in group_map:
                    group_map[part] = part
                return part

        left_var = resolve_var(left_str)
        right_var = resolve_var(right_str)
        if not left_var or not right_var:
            print(f"Thiếu biến cho {conn_str}")
            continue

        # Tạo tên biến mới
        new_name_base = f"{left_var}_{right_var}_{op}"
        new_name = new_name_base
        counter = 1
        while new_name in used_names:
            new_name = f"{new_name_base}_{counter}"
            counter += 1
        used_names.add(new_name)

        # Tạo phương trình
        if op == 'series':
            eq_str = f"Eq({new_name}, {left_var} + {right_var})"
        elif op == 'parallel':
            eq_str = f"Eq(1/{new_name}, 1/{left_var} + 1/{right_var})"
        else:
            continue

        specific_eqs.append({
            'id': f"conn_{conn_str.replace(' ', '_')}",
            'equation': eq_str,
            'variables': [new_name, left_var, right_var]
        })

        # Lưu mapping cho nhóm mới (để connection sau tham chiếu)
        group_key = f"{left_str},{right_str}"  # chuẩn hoá
        group_map[group_key] = new_name
        group_map[f"({left_str},{right_str})"] = new_name
        last_equivalent_var = new_name

    # 3. Thêm phương trình Ohm tổng nếu có thể
    if last_equivalent_var:
        # Tìm biến điện áp tổng (ưu tiên V, E, hoặc unit 'V')
        voltage_var = None
        for g in given:
            if g.get('unit') == 'V' or 'volt' in g.get('name', '').lower():
                voltage_var = g['symbol']
                break
        # Tìm biến dòng tổng (thường là unknown)
        current_var = None
        for u in unknown:
            if u.get('unit') == 'A' or 'current' in u.get('name', '').lower():
                current_var = u['symbol']
                break
        if voltage_var and current_var:
            specific_eqs.append({
                'id': 'ohms_law_total',
                'equation': f"Eq({voltage_var}, {current_var} * {last_equivalent_var})",
                'variables': [voltage_var, current_var, last_equivalent_var]
            })

    # 4. Thêm toàn bộ tools gốc (để BFS có thể dùng khi khớp tên biến)
    for tool in tools:
        specific_eqs.append({
            'id': tool['id'],
            'equation': tool['equation'],
            'variables': tool['variables']
        })

    return specific_eqs

# ============ HÀM INSTANTIATE MULTISTAGE CỦA BẠN (giữ nguyên) ============
def instantiate_multistage(tools, given, unknown, topology):
    # Code của bạn... (copy nguyên vào)
    indices = set()
    for item in given + unknown:
        sym = item['symbol']
        match = re.search(r'(\d+)$', sym)
        if match:
            indices.add(int(match.group(1)))
    if not indices:
        num_stages = 1
    else:
        num_stages = max(indices)

    base_vars = ['u', 'v', 'a', 't', 's']
    specific_eqs = []

    for i in range(1, num_stages + 1):
        var = lambda base: f"{base}{i}"
        for tool in tools:
            eq_str = tool['equation']
            mapping = {base: var(base) for base in base_vars if base in tool['variables']}
            new_eq_str = eq_str
            for base, indexed in mapping.items():
                new_eq_str = re.sub(r'\b' + base + r'\b', indexed, new_eq_str)
            specific_eqs.append({
                'id': f"{tool['id']}_stage{i}",
                'equation': new_eq_str,
                'variables': [mapping.get(v, v) for v in tool['variables']],
            })

    for i in range(1, num_stages):
        u_next = f"u{i+1}"
        v_curr = f"v{i}"
        specific_eqs.append({
            'id': f'link_v{i}_to_u{i+1}',
            'equation': f'Eq({u_next}, {v_curr})',
            'variables': [u_next, v_curr]
        })

    s_vars = [f"s{i}" for i in range(1, num_stages + 1)]
    s_total_eq = f"Eq(s_total, {' + '.join(s_vars)})"
    specific_eqs.append({
        'id': 'total_displacement',
        'equation': s_total_eq,
        'variables': ['s_total'] + s_vars
    })

    return specific_eqs

# ============ HÀM CHẠY TỰ ĐỘNG ============
def solve_physics_problem(question):
    # 1. Trích xuất JSON từ LLM (thay bằng hàm của bạn)
    json_str = problem_extracter(question)  # giả sử trả về chuỗi JSON
    data = json.loads(json_str)
    given = data['given']
    unknown = data['unknown']
    connections = data.get('connections', [])
    topology = data.get('topology', '')

    # 2. Lấy tools phù hợp
    tools = retrieve_relations(question, data['problem_type'], 5)

    # 3. Instantiate dựa trên connections
    if topology == 'multi_stage_linear_motion' or any('stage' in c for c in connections):
        specific_tools = instantiate_multistage(tools, given, unknown, topology)
    else:
        specific_tools = instantiate_general(given, unknown, connections, tools)

    # 4. Giải
    steps, values = forward_solve(given, unknown, specific_tools)
    return steps, values

# ============ TEST VỚI BÀI ĐIỆN CỦA BẠN ============
if __name__ == "__main__":
    q = q4

    # Mô phỏng problem_extracter và retrieve_relations bằng dữ liệu bạn đã in
    data_str = problem_extracter(q)
    data = json.loads(data_str)
    given = data['given']
    unknown = data['unknown']
    connections = data['connections']

    tools = retrieve_relations(q,"electricity")
    specific_tools = instantiate_general(given, unknown, connections, tools)
    print("=== Phương trình cụ thể ===")
    for t in specific_tools:
        print(t['equation'])

    steps, values = forward_solve(given, unknown, specific_tools)
    if steps:
        print("\n=== Các bước suy luận ===")
        for s in steps:
            print(f"{s['tool_id']}: {s['target']} = {s['value']}")
        print("\n=== Kết quả ===")
        for k, v in values.items():
            print(f"{k} = {v}")
    else:
        print("Không giải được.")

Không parse được connection: R1 and R2 in parallel
Thiếu biến cho (R1,R2) parallel R3
=== Phương trình cụ thể ===
Eq(V, I * R)
Eq(1/R_total, 1/R1 + 1/R2)
Eq(R_total, R1 + R2)
Eq(I_x, I_total * (R_total / R_x))
Eq(V_x, V_total * (R_x / R_total))
Không thể tìm được các biến: ['I', 'P3']
Không giải được.


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:5])

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/nam1706 []
/kaggle/input/datasets/nam1706/phybasicextracting ['phyExtractingProblemPrompt.txt']
/kaggle/input/datasets/nam1706/phyextraction []
/kaggle/input/datasets/nam1706/phyextraction/phyExtractionPrompt ['EXTRACT_ELECTRICITY_PROMPT.txt', 'EXTRACT_KINEMATICS_SHMPROMPT.txt', 'EXTRACT_DYNAMICS_PROMPT.txt', 'EXTRACT_THERMODYNAMICS_PROMPT.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-0 ['phyExtractingProblemPrompt.txt']
/kaggle/input/datasets/nam1706/phyextraction-vfinal []
/kaggle/input/datasets/nam1706/phyextraction-vfinal/phyExtractionPrompt ['EXTRACT_ELECTRICITY_PROMPT.txt', 'EXTRACT_KINEMATICS_SHMPROMPT.txt', 'EXTRACT_DYNAMICS_PROMPT.txt', 'EXTRACT_THERMODYNAMICS_PROMPT.txt']
/kaggle/input/datasets/nam1706/prompt-v1-0 ['prompt.txt']
